# Lokaler Diagnosetest: NewsBERT-germ-210m

Laedt das Modell von HuggingFace, reproduziert den exakt selben Test-Split
wie im Training und evaluiert die Performance. Laeuft auf MacBook M1 (CPU).

**Erwartetes Ergebnis (wenn Modell korrekt):** F1 Macro ~0.84
**Wenn Modell fehlerhaft:** F1 Macro <0.10

In [1]:
# === SETUP ===
import sys
from pathlib import Path

PIPELINE_DIR = str(Path.cwd().parent) if Path.cwd().name == "train_final_model" else str(Path.cwd() / "Python" / "classification_pipeline")
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

import importlib
import pipeline_utils as pu
importlib.reload(pu)

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
)
from transformers.modeling_rope_utils import ROPE_INIT_FUNCTIONS

from huggingface_hub import login
login()

print("Setup OK.")

Setup OK.


In [3]:
# === KONFIGURATION ===
MODEL_REPO = "Zorryy/NewsBERT-germ-210m"
RANDOM_SEED = 42
TEST_PER_CLASS = 30
BATCH_SIZE = 8
MAX_LENGTH = 2048

ALL_LABELS = [
    "Klima / Energie", "Zuwanderung", "Renten", "Soziales Gef\u00e4lle",
    "AfD/Rechte", "Arbeitslosigkeit", "Wirtschaftslage", "Politikverdruss",
    "Gesundheitswesen, Pflege", "Kosten/L\u00f6hne/Preise",
    "Ukraine/Krieg/Russland", "Bundeswehr/Verteidigung", "Andere",
]

label2id = {label: idx for idx, label in enumerate(ALL_LABELS)}
id2label = {idx: label for idx, label in enumerate(ALL_LABELS)}

def safe_col_name(label):
    return ("score_" + label
            .replace(" / ", "_").replace("/", "_").replace(", ", "_")
            .replace(" ", "_")
            .replace("\u00e4", "ae").replace("\u00f6", "oe").replace("\u00fc", "ue"))

SCORE_COLUMNS = [safe_col_name(l) for l in ALL_LABELS]

OUTPUT_CSV = Path(PIPELINE_DIR) / "performance_reports" / "local_test_results.csv"
device = torch.device("cpu")

print(f"Device: {device}")
print(f"Output: {OUTPUT_CSV}")

Device: cpu
Output: /Users/zorbeyozcan/news_articles_classification_thesis/Python/classification_pipeline/performance_reports/local_test_results.csv


In [4]:
# === DATASET LADEN & TEST-SPLIT REPRODUZIEREN ===
# Exakt derselbe Split wie in Cell 4 von train_final_eurobert_210m.ipynb

ds = load_dataset(pu.DATASET_ID)
train_hf = ds["train"].to_pandas()
test_hf = ds["test"].to_pandas()
all_labelled = pd.concat([train_hf, test_hf], ignore_index=True)
print(f"Gelabelte Artikel gesamt: {len(all_labelled)}")

np.random.seed(RANDOM_SEED)

test_indices = []
rest_indices = []

for label in ALL_LABELS:
    label_mask = all_labelled["label"] == label
    label_indices = all_labelled[label_mask].index.tolist()
    n_total = len(label_indices)
    if n_total < 60:
        n_test = n_total // 2
        print(f"  {label}: nur {n_total} Artikel -> {n_test} fuer Test")
    else:
        n_test = TEST_PER_CLASS
    np.random.shuffle(label_indices)
    test_indices.extend(label_indices[:n_test])
    rest_indices.extend(label_indices[n_test:])

test_df = all_labelled.loc[test_indices].reset_index(drop=True)

print(f"\nTest-Split: {len(test_df)} Artikel")
print(f"Klassen-Verteilung:")
for label in ALL_LABELS:
    n = len(test_df[test_df["label"] == label])
    print(f"  {label:<30} {n:>3}")

Gelabelte Artikel gesamt: 2360
  Politikverdruss: nur 50 Artikel -> 25 fuer Test

Test-Split: 385 Artikel
Klassen-Verteilung:
  Klima / Energie                 30
  Zuwanderung                     30
  Renten                          30
  Soziales Gefälle                30
  AfD/Rechte                      30
  Arbeitslosigkeit                30
  Wirtschaftslage                 30
  Politikverdruss                 25
  Gesundheitswesen, Pflege        30
  Kosten/Löhne/Preise             30
  Ukraine/Krieg/Russland          30
  Bundeswehr/Verteidigung         30
  Andere                          30


In [5]:
# === MODELL LADEN + SANITY CHECK ===

# EuroBERT RoPE Fix
def _default_rope_init(config, device=None, **kwargs):
    base = getattr(config, "rope_theta", 10000.0)
    partial_rotary_factor = getattr(config, "partial_rotary_factor", 1.0)
    head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
    dim = int(head_dim * partial_rotary_factor)
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.int64).float().to(device) / dim))
    return inv_freq, 1.0

ROPE_INIT_FUNCTIONS["default"] = _default_rope_init

print(f"Lade Modell: {MODEL_REPO}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_REPO, trust_remote_code=True, attn_implementation="eager",
)
model = model.to(device)
model.eval()

# Label-Mapping validieren
model_labels = [model.config.id2label[i] for i in range(len(model.config.id2label))]
assert model_labels == ALL_LABELS, f"Label-Mismatch!\n  Modell: {model_labels}\n  Erwartet: {ALL_LABELS}"
print(f"Labels: {len(model.config.id2label)} (OK)")

# Sanity Check
SANITY_TESTS = [
    ("Die Bundesregierung plant neue Massnahmen zur Reduzierung der CO2-Emissionen "
     "und zum Ausbau erneuerbarer Energien im Rahmen des Klimaschutzgesetzes.",
     "Klima / Energie"),
    ("Die AfD hat bei den letzten Umfragen zugelegt und liegt nun bei 22 Prozent. "
     "Die Partei fordert eine haertere Migrationspolitik.",
     "AfD/Rechte"),
    ("Die Rentenreform wird heftig diskutiert. Viele Rentner fuerchten Kuerzungen "
     "ihrer Bezuege. Das Renteneintrittsalter soll angehoben werden.",
     "Renten"),
    ("Der Krieg in der Ukraine geht weiter. Russland hat erneut Raketenangriffe "
     "auf ukrainische Staedte gestartet. Die NATO beraet ueber weitere Hilfen.",
     "Ukraine/Krieg/Russland"),
    ("Die Bundeswehr braucht dringend neue Ausruestung und mehr Soldaten. "
     "Das Sondervermoegen fuer die Verteidigung wird aufgestockt.",
     "Bundeswehr/Verteidigung"),
]

print(f"\n{'='*70}")
print(f"  SANITY CHECK")
print(f"{'='*70}")

sanity_passed = 0
for text, expected in SANITY_TESTS:
    inputs = tokenizer(text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits.float(), dim=-1)
    pred_id = probs.argmax(dim=-1).item()
    pred_label = id2label[pred_id]
    score = probs[0, pred_id].item()
    ok = pred_label == expected and score >= 0.40
    if ok:
        sanity_passed += 1
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] Erwartet: {expected:<30} -> {pred_label} ({score:.4f})")

print(f"\n  Ergebnis: {sanity_passed}/{len(SANITY_TESTS)}")

Lade Modell: Zorryy/NewsBERT-germ-210m...


--- Logging error ---
Traceback (most recent call last):
  File "/Users/zorbeyozcan/anaconda3/lib/python3.11/logging/__init__.py", line 1110, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/Users/zorbeyozcan/anaconda3/lib/python3.11/logging/__init__.py", line 953, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/Users/zorbeyozcan/anaconda3/lib/python3.11/logging/__init__.py", line 687, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/Users/zorbeyozcan/anaconda3/lib/python3.11/logging/__init__.py", line 377, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/zorbeyozcan/news_articles_classification_thesis/Python/classification_pipeline/.venv/lib/python3.11/site-packages/

Loading weights:   0%|          | 0/114 [00:00<?, ?it/s]

Labels: 13 (OK)

  SANITY CHECK
  [PASS] Erwartet: Klima / Energie                -> Klima / Energie (0.9105)
  [PASS] Erwartet: AfD/Rechte                     -> AfD/Rechte (0.8688)
  [PASS] Erwartet: Renten                         -> Renten (0.9245)
  [PASS] Erwartet: Ukraine/Krieg/Russland         -> Ukraine/Krieg/Russland (0.9178)
  [PASS] Erwartet: Bundeswehr/Verteidigung        -> Bundeswehr/Verteidigung (0.9139)

  Ergebnis: 5/5


In [6]:
# === INFERENCE AUF TEST-SPLIT ===
print(f"Inference auf {len(test_df)} Artikeln (Batch={BATCH_SIZE}, CPU)...")
print("(Dauert auf CPU einige Minuten)\n")

texts = test_df["text"].fillna("").tolist()
hf_dataset = Dataset.from_dict({"text": texts})
hf_dataset = hf_dataset.map(
    lambda x: tokenizer(x["text"], max_length=MAX_LENGTH, truncation=True),
    batched=True, batch_size=256, remove_columns=["text"],
)
hf_dataset.set_format("torch", columns=["input_ids", "attention_mask"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")
dataloader = DataLoader(
    hf_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator,
)

all_predictions = []
all_scores = []

with torch.no_grad():
    for batch in tqdm(dataloader, desc="Inference"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits.float(), dim=-1).numpy()
        pred_ids = probs.argmax(axis=-1)
        all_predictions.extend(id2label[int(i)] for i in pred_ids)
        all_scores.extend(probs.tolist())

print(f"\nInference abgeschlossen: {len(all_predictions)} Predictions")

Inference auf 385 Artikeln (Batch=8, CPU)...
(Dauert auf CPU einige Minuten)



Map:   0%|          | 0/385 [00:00<?, ? examples/s]

Inference: 100%|██████████| 49/49 [11:47<00:00, 14.45s/it]


Inference abgeschlossen: 385 Predictions


In [7]:
# === EVALUATION ===
true_labels = test_df["label"].tolist()
metrics = pu.evaluate(true_labels, all_predictions, labels=ALL_LABELS, experiment_name="local_test")
pu.print_metrics(metrics, "NewsBERT-germ-210m — Lokaler Test (CPU)")

  NewsBERT-germ-210m — Lokaler Test (CPU)

  F1 Macro:           0.8204
  F1 Weighted:        0.8207
  Precision Macro:    0.8387
  Precision Weighted: 0.8379
  Recall Macro:       0.8169
  Recall Weighted:    0.8182
  Accuracy:           0.8182

  Per-Class Metrics:
                   Label  Precision   Recall       F1  Support
         Klima / Energie   0.787879 0.866667 0.825397       30
             Zuwanderung   0.838710 0.866667 0.852459       30
                  Renten   0.857143 0.800000 0.827586       30
        Soziales Gefälle   0.678571 0.633333 0.655172       30
              AfD/Rechte   0.793103 0.766667 0.779661       30
        Arbeitslosigkeit   1.000000 0.700000 0.823529       30
         Wirtschaftslage   0.560000 0.933333 0.700000       30
         Politikverdruss   0.900000 0.720000 0.800000       25
Gesundheitswesen, Pflege   0.903226 0.933333 0.918033       30
     Kosten/Löhne/Preise   0.920000 0.766667 0.836364       30
  Ukraine/Krieg/Russland   0.935484 0.9

In [8]:
# === CSV SPEICHERN & ZUSAMMENFASSUNG ===
scores_array = np.array(all_scores)
test_df["prediction"] = all_predictions
test_df["prediction_score"] = scores_array.max(axis=1)
for i, col_name in enumerate(SCORE_COLUMNS):
    test_df[col_name] = scores_array[:, i]

output_cols = ["id", "headline", "label", "prediction", "prediction_score"] + SCORE_COLUMNS
output_cols = [c for c in output_cols if c in test_df.columns]
output_df = test_df[output_cols]

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
output_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"CSV gespeichert: {OUTPUT_CSV}")
print(f"Zeilen: {len(output_df)}")

print(f"\n{'=' * 70}")
print(f"  ZUSAMMENFASSUNG")
print(f"{'=' * 70}")
print(f"  Modell:          {MODEL_REPO}")
print(f"  Device:          {device}")
print(f"  Test-Artikel:    {len(test_df)}")
print(f"  Sanity Check:    {sanity_passed}/{len(SANITY_TESTS)}")
print(f"  F1 Macro:        {metrics['f1_macro']:.4f}")
print(f"  F1 Weighted:     {metrics['f1_weighted']:.4f}")
print(f"  Accuracy:        {metrics['accuracy']:.4f}")
print(f"  Mean Confidence: {scores_array.max(axis=1).mean():.4f}")
print(f"{'=' * 70}")

if metrics["f1_macro"] > 0.70:
    print("\n  -> Modell performt gut lokal. Problem liegt an der Colab-Umgebung.")
else:
    print("\n  -> Modell performt SCHLECHT. Das Modell auf HuggingFace ist fehlerhaft.")
    print("     -> Modell muss neu trainiert und gepusht werden.")

CSV gespeichert: /Users/zorbeyozcan/news_articles_classification_thesis/Python/classification_pipeline/performance_reports/local_test_results.csv
Zeilen: 385

  ZUSAMMENFASSUNG
  Modell:          Zorryy/NewsBERT-germ-210m
  Device:          cpu
  Test-Artikel:    385
  Sanity Check:    5/5
  F1 Macro:        0.8204
  F1 Weighted:     0.8207
  Accuracy:        0.8182
  Mean Confidence: 0.8862

  -> Modell performt gut lokal. Problem liegt an der Colab-Umgebung.
